# Curvature and smoothing, step by step

Companion to [W10 §6](../README.md#6-curvature-the-angle-defect) and
[§7](../README.md#7-smoothing-laplacian-and-taubin). You will:

1. measure the curvature at one vertex by adding up its angles;
2. check Gauss–Bonnet on a whole sphere;
3. smooth a noisy sphere one step by hand;
4. watch thirty rounds of Laplacian smoothing shrink it, and Taubin not.

**Setup.** Run this cell and move on. It lets the notebook find the drawing helpers
(`viz`) and this week's algorithms (`examples/`), whether it sits in the lecture
folder or is your copy in `MyWork/`.

In [ ]:
import sys
import pathlib

HERE = pathlib.Path.cwd()
SHARED = next(p / "Lecture" / "shared" for p in [HERE, *HERE.parents] if (p / "Lecture" / "shared" / "viz.py").exists())
sys.path[:0] = [str(SHARED), str(SHARED.parent / "Lecture_10" / "examples")]

## Step 1 — The angles around one vertex

A flat grid of four squares: the angles around the middle vertex add up to 360°.
Lift that vertex 0.6 m, like a tent pole. The four faces tilt, their angles at the top
shrink, and the sum falls short of 360°. The shortfall is the **angle defect**: the
curvature at that vertex. Positive means dome-like.

In [ ]:
import math

from compas.datastructures import Mesh

import curvature
from viz import draw_mesh

tent = Mesh.from_meshgrid(dx=2, nx=2)
top = next(v for v in tent.vertices() if not tent.is_vertex_on_boundary(v))
tent.vertex_attribute(top, "z", 0.6)

angles = [math.degrees(curvature.face_angle(tent, f, top)) for f in tent.vertex_faces(top)]
print("angles at the top:", [round(a, 1) for a in angles])
print(f"sum {sum(angles):.1f} deg -> angle defect {360 - sum(angles):.1f} deg")
draw_mesh(tent, "a lifted vertex", view="iso", highlight_vertices=[top])

## Step 2 — Gauss–Bonnet on a whole sphere

Add the defects of every vertex of a closed mesh and you get 2π × its Euler number: 4π
for anything shaped like a ball, however it is meshed. Check it on a sphere of 24 × 16
faces. The ratio printed must be 4 exactly (up to rounding).

In [ ]:
from compas.geometry import Sphere

ball = Mesh.from_shape(Sphere(radius=1.0), u=24, v=16)
total = curvature.total_curvature(ball)
print(f"sum of defects / pi = {total / math.pi:.6f}   euler = {ball.euler()}")

## Step 3 — One smoothing step, by hand

`noisy` shakes every vertex up to 6 cm, a stand-in for a 3D scan. One Laplacian step
moves each vertex halfway to the average of its neighbours. All new positions are
computed from the old ones first (`new`), and only then written. Compare roughness and
radius before and after.

In [ ]:
import smoothing

rough = smoothing.noisy(ball, 0.06, seed=0)
print(f"before: radius {smoothing.mean_radius(rough):.4f}, roughness {smoothing.roughness(rough):.4f}")

new = {}
for v in rough.vertices():
    neighbours = [rough.vertex_coordinates(n) for n in rough.vertex_neighbors(v)]
    centre = [sum(c[i] for c in neighbours) / len(neighbours) for i in range(3)]
    p = rough.vertex_coordinates(v)
    new[v] = [p[i] + 0.5 * (centre[i] - p[i]) for i in range(3)]
for v, xyz in new.items():
    rough.vertex_attributes(v, "xyz", xyz)

print(f"after:  radius {smoothing.mean_radius(rough):.4f}, roughness {smoothing.roughness(rough):.4f}")

## Step 4 — Thirty rounds: Laplacian shrinks, Taubin does not

The tested functions, 0 to 30 rounds, measuring the radius every third round.
Laplacian keeps pulling every vertex inward. Taubin adds a slightly larger step back
outward after each one, so the noise goes and the size stays.

In [ ]:
from viz import chart

start = smoothing.noisy(ball, 0.06, seed=0)
rounds = range(0, 31, 3)
laplacian = [smoothing.mean_radius(smoothing.laplacian_smooth(start, rounds=r)) for r in rounds]
taubin = [smoothing.mean_radius(smoothing.taubin_smooth(start, rounds=r)) for r in rounds]
print(f"after 30 rounds: Laplacian {laplacian[-1]:.3f}, Taubin {taubin[-1]:.3f}")
chart({"Laplacian": laplacian, "Taubin": taubin}, "mean radius over 30 rounds", "rounds (x3)")

The three spheres side by side: the noisy input (grey), Laplacian after 30 rounds
(red, smaller), Taubin after 30 rounds (blue, the same size). Drag to orbit.

In [ ]:
from compas.colors import Color
from compas.geometry import Translation

from viz import view3d

view3d((start, {"facecolor": Color.grey().lightened(50)}),
       (smoothing.laplacian_smooth(start, rounds=30).transformed(Translation.from_vector([2.5, 0, 0])),
        {"facecolor": Color.from_hex("#ffa8a8")}),
       (smoothing.taubin_smooth(start, rounds=30).transformed(Translation.from_vector([5, 0, 0])),
        {"facecolor": Color.from_hex("#74c0fc")}))

## Your turn

1. In Step 1, lift the vertex to 2.0. Does the defect grow? Push it down to -0.6: what
   changes, and why is the sign the same?
2. In Step 3, write each new position straight into the mesh instead of collecting
   `new` first. Does the result change? Why?
3. Try `smoothing.taubin_smooth(start, mu=-0.5)`. What does it say, and why?